# SeaFour Retrieval Engine Submission Notebook


This notebook also serves as the competition report, with narrative context alongside the runnable pipeline.
This notebook is written as a reproducible engineering report for the retrieval competition. It keeps the original experimental intent of the submission notebook, but reorganizes the work into a clearer pipeline: data loading, preprocessing, index construction, retrieval, evaluation, and submission generation.

The notebook implements three first-stage retrieval methods:
- **TF-IDF** for a lightweight sparse lexical baseline
- **BM25+** for stronger lexical ranking with length normalization
- **Embedding retrieval** for dense semantic search with Sentence-Transformers

The default submission path still uses **embedding retrieval** plus the existing category classifier, because that is the strongest practical default for this dataset.

If a fresh environment is missing packages, install them before running the notebook:

```python
%pip install -q rank_bm25 sentence-transformers scikit-learn
```


## Retrieval Engine Overview

The end-to-end pipeline in this notebook is:

1. **Load competition files**: documents, train queries, test queries, training ground truth, and the sample submission template.
2. **Normalize text consistently**: merge the relevant text fields, standardize separators and whitespace, lowercase the text, and build a single `content` field for each document and query.
3. **Build retrieval artifacts**:
   - TF-IDF document-term matrix for lexical cosine search
   - BM25+ index over tokenized documents
   - Dense document embeddings for semantic retrieval
4. **Retrieve top-k candidates** for each query using the chosen first-stage retriever.
5. **Score the system offline** on the training queries using Recall@K, Precision@K, MRR@K, and category accuracy.
6. **Generate the final Kaggle submission** for the test queries.

This notebook uses direct first-stage retrieval only: retrieve top-k documents, evaluate on the training queries, and write the submission directly from the selected retriever.


## Why Embeddings Are the Primary Retrieval Method

Embedding retrieval is the best default method in this notebook because it captures **semantic similarity**, not only exact token overlap.

In practice, embeddings win here when:
- the query paraphrases the relevant document instead of reusing the same words
- users describe the same issue with different wording, abbreviations, or syntax
- the corpus is heterogeneous and contains multiple technical domains with overlapping vocabulary
- retrieval needs to stay robust when queries and documents use different wording for the same issue

Why embeddings are stronger than the lexical baselines in this notebook:
- **Better robustness to wording mismatch**: a query about a concept can still retrieve documents that do not share the same surface form.
- **Higher recall under semantic mismatch**: this matters a lot for support-style text, forum questions, and troubleshooting descriptions.
- **Better candidate generation**: dense retrieval is often the strongest first-stage retriever when wording mismatch is common.
- **More stable default behavior** when queries and relevant documents do not use the exact same vocabulary.

Honest limits of embeddings:
- they require more compute and more memory than simple lexical methods
- they can retrieve items that are semantically related but still not truly relevant
- they are not always best for **exact-match retrieval**, especially for rare identifiers, product codes, version strings, names, or error messages

In those exact-match cases, TF-IDF or BM25+ can still be valuable because lexical overlap is the signal you care about.


## Choosing `top_k`

`top_k` is the number of documents returned per query by the first-stage retriever.

Why `top_k` matters:
- **Too small**: the retriever may miss relevant documents, which hurts recall.
- **Too large**: recall usually improves, but precision drops, latency grows, and memory use increases.

Practical tradeoffs:

| Setting | Usually helps | Usually hurts |
|---|---|---|
| Small `top_k` | Precision, latency, memory footprint | Recall |
| Large `top_k` | Recall | Precision, latency, memory footprint |

Metric impact:
- **Recall@K** usually increases as `top_k` grows because more relevant documents can appear in the candidate list.
- **Precision@K** often decreases because the tail of the ranking includes more non-relevant documents.
- **MRR@K** changes less once the first relevant document is already near the top.
- **Latency** increases because more scores must be computed or sorted.
- **Memory** increases when storing larger score blocks and larger submission payloads.

In this notebook, `top_k` is also the final number of retrieved documents used for evaluation or submission.


## When to Use Baseline / Lexical / Embedding Retrieval

Use the method that matches the failure mode you expect most often.

| Method | Prefer it when | Less suitable when |
|---|---|---|
| **TF-IDF** | You want a simple, fast baseline; documents are short; overlap in rare terms is informative | Queries and documents use different phrasing or synonyms |
| **BM25+** | Exact words matter, document lengths vary, and you want a stronger lexical baseline than TF-IDF | Semantic mismatch is common |
| **Embeddings** | Queries paraphrase documents, recall matters, and the corpus is heterogeneous | You are extremely latency-constrained or exact IDs / codes dominate relevance |

Practical selection guidance:
- **Small datasets**: TF-IDF or BM25+ may be good enough and are easier to debug.
- **Large datasets**: embeddings often pay off because semantic mismatch becomes more common and lexical recall becomes brittle.
- **High semantic mismatch**: prefer embeddings.
- **Strict latency budget**: prefer lexical methods unless dense retrieval is well optimized or precomputed.
- **Recall more important than precision**: prefer embeddings and a larger candidate pool.
- **Precision at very small K matters more**: BM25+ can be competitive when exact wording is highly reliable.


In [1]:
# Imports, paths, and experiment configuration.
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal, Sequence, TypedDict
import csv
import hashlib
import json
import os
import pickle
import re
import time

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 5)

ModelName = Literal["tfidf", "bm25", "embedding"]


class RetrievalResult(TypedDict):
    """One ranked retrieval result for a single query."""

    query_id: str
    relevant_docs: list[str]


class GroundTruthEntry(TypedDict):
    """Ground-truth annotations for a single training query."""

    relevant_doc_ids: set[str]
    total_relevant_docs: int
    category: str | None


@dataclass(frozen=True)
class RuntimePaths:
    """Resolved filesystem locations for the active notebook runtime."""

    runtime_env: Literal["colab", "kaggle", "local"]
    project_dir: Path
    work_dir: Path
    data_dir: Path
    cache_dir: Path
    output_path: Path


def detect_runtime_environment() -> Literal["colab", "kaggle", "local"]:
    """Detect whether the notebook runs in Colab, Kaggle, or a local environment."""
    try:
        import google.colab  # type: ignore  # noqa: F401

        return "colab"
    except Exception:
        if Path("/kaggle/input").exists():
            return "kaggle"
        return "local"


def find_kaggle_data_dir() -> Path | None:
    """Search `/kaggle/input` for the folder that contains the competition JSON files."""
    for dirname, _, filenames in os.walk("/kaggle/input"):
        if "docs.json" in filenames:
            return Path(dirname)
    return None


def find_colab_project_dir(project_name: str = "retrieval_project") -> Path | None:
    """Locate the project folder on Google Drive when running in Colab."""
    drive_candidates = [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/Shareddrives"),
    ]

    for drive_root in drive_candidates:
        if not drive_root.exists():
            continue

        direct_candidate = drive_root / project_name
        if (direct_candidate / "data" / "docs.json").exists():
            return direct_candidate

        for candidate in drive_root.rglob(project_name):
            if candidate.is_dir() and (candidate / "data" / "docs.json").exists():
                return candidate

    return None


def find_local_project_data_dir() -> Path | None:
    """Find the `data/` directory near the current working directory."""
    for root in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        data_dir = root / "data"
        if (data_dir / "docs.json").exists():
            return data_dir
    return None


def resolve_runtime_paths(output_filename: str = "solutions_SeaFour.csv") -> RuntimePaths:
    """Resolve project, data, cache, and output paths for the active runtime."""
    runtime_env = detect_runtime_environment()
    project_dir: Path | None = None
    work_dir = Path.cwd()

    if runtime_env == "colab":
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive", force_remount=False)
        project_dir = find_colab_project_dir("retrieval_project")
        if project_dir is None:
            raise FileNotFoundError(
                "Google Drive is mounted, but `retrieval_project/data/docs.json` was not found."
            )
        os.chdir(project_dir)
        work_dir = project_dir
        data_dir = project_dir / "data"
    elif runtime_env == "kaggle":
        data_dir = find_kaggle_data_dir()
        if data_dir is None:
            raise FileNotFoundError(
                "Kaggle environment detected, but `docs.json` was not found under /kaggle/input."
            )
        project_dir = Path.cwd()
    else:
        data_dir = find_local_project_data_dir()
        if data_dir is None:
            raise FileNotFoundError(
                "Could not find `data/docs.json` near the current working directory."
            )
        project_dir = data_dir.parent
        work_dir = project_dir
        os.chdir(work_dir)

    cache_dir = work_dir / "cache"
    output_path = work_dir / output_filename
    return RuntimePaths(
        runtime_env=runtime_env,
        project_dir=project_dir or work_dir,
        work_dir=work_dir,
        data_dir=data_dir,
        cache_dir=cache_dir,
        output_path=output_path,
    )


# High-level experiment settings.
FINAL_MODEL: ModelName = "embedding"
EVALUATION_MODELS: tuple[ModelName, ...] = ("embedding",)
EVALUATION_TOP_KS = [ 7500 ]#tuple[int, ...] = (7_500)#,12_500, 75_000)
SUBMIT_TOP_K = 7_500
ENABLE_CATEGORY_FILTER = True
DOMINANT_CATEGORY_TOP_N = 20

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
EMBEDDING_BATCH_SIZE = 256
EMBEDDING_QUERY_CHUNK_SIZE = 32

DOCUMENT_TEXT_COLUMNS: tuple[str, ...] = ("title", "text", "tags")
RETRIEVAL_QUERY_COLUMNS: tuple[str, ...] = ("title", "text")
CLASSIFIER_QUERY_COLUMNS: tuple[str, ...] = ("title", "text", "tags")
CLASSIFIER_USE_QUERY_TAGS = True

NORMALIZATION_CONFIG = {
    "lowercase": True,
    "replace_separators": True,
    "separator_chars": "-_/",
    "collapse_whitespace": True,
    "strip": True,
}
TOKEN_PATTERN = r"[a-z0-9]+"

TFIDF_CONFIG = {
    "lowercase": True,
    "ngram_range": (1, 2),
    "min_df": 2,
}
CLASSIFIER_TFIDF_CONFIG = {
    "lowercase": True,
    "ngram_range": (1, 2),
    "min_df": 2,
}
BM25_CONFIG = {
    "k1": 1.5,
    "b": 0.75,
    "delta": 1.0,
}

ENABLE_EMBEDDING_CACHE = True
ENABLE_CLASSIC_CACHE = True

PATHS = resolve_runtime_paths()
MODEL_CACHE_DIR = PATHS.cache_dir / "sentence_transformers"
EMBEDDING_CACHE_DIR = PATHS.cache_dir / "embeddings"
TFIDF_CACHE_DIR = PATHS.cache_dir / "tfidf"
BM25_CACHE_DIR = PATHS.cache_dir / "bm25"
CLASSIFIER_CACHE_DIR = PATHS.cache_dir / "classifier"

for cache_path in [
    PATHS.cache_dir,
    MODEL_CACHE_DIR,
    EMBEDDING_CACHE_DIR,
    TFIDF_CACHE_DIR,
    BM25_CACHE_DIR,
    CLASSIFIER_CACHE_DIR,
]:
    cache_path.mkdir(parents=True, exist_ok=True)

print(f"Runtime environment: {PATHS.runtime_env}")
print(f"Project directory  : {PATHS.project_dir}")
print(f"Working directory  : {PATHS.work_dir}")
print(f"Data directory     : {PATHS.data_dir}")
print(f"Output path        : {PATHS.output_path}")


Runtime environment: local
Project directory  : /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project
Working directory  : /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project
Data directory     : /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project/data
Output path        : /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project/solutions_SeaFour.csv


## Data Loading

This section loads the raw competition files and validates the expected schema early. The checks are intentionally strict so notebook failures happen close to the source of the problem instead of later in the retrieval pipeline.


In [2]:
def require_columns(frame: pd.DataFrame, required_columns: Sequence[str], frame_name: str) -> None:
    """Raise a clear error if a dataframe is missing required columns."""
    missing_columns = [column for column in required_columns if column not in frame.columns]
    if missing_columns:
        raise ValueError(f"{frame_name} is missing required columns: {missing_columns}")


def ensure_unique_ids(frame: pd.DataFrame, frame_name: str) -> None:
    """Ensure that the `id` column exists and does not contain duplicates."""
    require_columns(frame, ["id"], frame_name)
    if not frame["id"].astype(str).is_unique:
        raise ValueError(f"{frame_name} contains duplicate ids, which would break retrieval output mapping.")


def load_json_frame(path: Path, frame_name: str) -> pd.DataFrame:
    """Load one JSON competition file into a dataframe."""
    if not path.exists():
        raise FileNotFoundError(f"{frame_name} file not found: {path}")
    return pd.read_json(path)


docs_raw_df = load_json_frame(PATHS.data_dir / "docs.json", "Documents")
train_queries_raw_df = load_json_frame(PATHS.data_dir / "queries_train.json", "Train queries")
test_queries_raw_df = load_json_frame(PATHS.data_dir / "queries_test.json", "Test queries")
sample_submission_path = PATHS.data_dir / "submission.csv"
ground_truth_path = PATHS.data_dir / "qgts_train.json"

sample_submission_df = pd.read_csv(sample_submission_path)

require_columns(docs_raw_df, ["id", "title", "text", "tags", "category"], "Documents")
require_columns(train_queries_raw_df, ["id", "title", "text", "tags", "category"], "Train queries")
require_columns(test_queries_raw_df, ["id", "title", "text", "tags"], "Test queries")
require_columns(sample_submission_df, ["query_id", "relevant_doc_ids", "category"], "Sample submission")

ensure_unique_ids(docs_raw_df, "Documents")
ensure_unique_ids(train_queries_raw_df, "Train queries")
ensure_unique_ids(test_queries_raw_df, "Test queries")

print(f"Documents      : {len(docs_raw_df):,}")
print(f"Train queries  : {len(train_queries_raw_df):,}")
print(f"Test queries   : {len(test_queries_raw_df):,}")
print(f"Sample rows    : {len(sample_submission_df):,}")


Documents      : 216,041
Train queries  : 327
Test queries   : 141
Sample rows    : 141


## Preprocessing

Retrieval quality depends heavily on text normalization. The goal here is not aggressive linguistic processing; it is simple, consistent engineering hygiene. We create a single `content` field per row, keep the logic shared across methods, and preserve the original design choice that retrieval uses only query title and text, while the classifier may also use query tags.


In [3]:
_TOKEN_RE = re.compile(TOKEN_PATTERN)
_WHITESPACE_RE = re.compile(r"\s+")
_SEPARATOR_RE = re.compile(f"[{re.escape(NORMALIZATION_CONFIG['separator_chars'])}]")


def value_to_text(value: Any) -> str:
    """Convert raw dataframe values into plain text.

    Args:
        value: A scalar, list-like, or missing value from a dataframe cell.

    Returns:
        A string representation suitable for text normalization.
    """
    if value is None:
        return ""
    if isinstance(value, (list, tuple, set)):
        return " ".join(str(item) for item in value)
    if pd.isna(value):
        return ""
    return str(value)


def normalize_text(text: Any) -> str:
    """Apply the shared normalization policy used across retrievers.

    Args:
        text: Raw text or text-like content.

    Returns:
        Normalized text with separators replaced, whitespace collapsed, and casing standardized.
    """
    if text is None:
        cleaned_text = ""
    elif not isinstance(text, str) and pd.isna(text):
        cleaned_text = ""
    else:
        cleaned_text = str(text)

    if NORMALIZATION_CONFIG["replace_separators"]:
        cleaned_text = _SEPARATOR_RE.sub(" ", cleaned_text)
    if NORMALIZATION_CONFIG["lowercase"]:
        cleaned_text = cleaned_text.lower()
    if NORMALIZATION_CONFIG["collapse_whitespace"]:
        cleaned_text = _WHITESPACE_RE.sub(" ", cleaned_text)
    if NORMALIZATION_CONFIG["strip"]:
        cleaned_text = cleaned_text.strip()
    return cleaned_text


def build_content_frame(frame: pd.DataFrame, text_columns: Sequence[str]) -> pd.DataFrame:
    """Build a dataframe with a normalized `content` column.

    Args:
        frame: Input dataframe that must contain `id` and the requested text columns.
        text_columns: Columns to concatenate into the normalized content field.

    Returns:
        A copy of the input dataframe with string ids and a new `content` column.
    """
    require_columns(frame, ["id"], "Input frame")
    output_frame = frame.copy()
    text_parts: list[list[str]] = []

    for column in text_columns:
        if column in output_frame.columns:
            text_parts.append(output_frame[column].map(value_to_text).tolist())
        else:
            text_parts.append([""] * len(output_frame))

    merged_text = [" ".join(parts) for parts in zip(*text_parts)]
    output_frame["content"] = [normalize_text(text) for text in merged_text]
    output_frame["id"] = output_frame["id"].astype(str)
    return output_frame


def tokenize(text: str) -> list[str]:
    """Tokenize normalized text for lexical retrieval."""
    return _TOKEN_RE.findall(normalize_text(text))


def build_query_classifier_frame(query_frame: pd.DataFrame, include_tags: bool = CLASSIFIER_USE_QUERY_TAGS) -> pd.DataFrame:
    """Build the text view used by the category classifier.

    Args:
        query_frame: Raw query dataframe.
        include_tags: Whether to include the `tags` column when available.

    Returns:
        A dataframe with a classifier-oriented `content` column.
    """
    columns = ["title", "text"]
    if include_tags and "tags" in query_frame.columns:
        columns.append("tags")
    return build_content_frame(query_frame, columns)


docs_df = build_content_frame(docs_raw_df, DOCUMENT_TEXT_COLUMNS)
train_queries_df = build_content_frame(train_queries_raw_df, RETRIEVAL_QUERY_COLUMNS)
test_queries_df = build_content_frame(test_queries_raw_df, RETRIEVAL_QUERY_COLUMNS)

docs_classifier_df = docs_df[["id", "content", "category"]].copy()
train_queries_classifier_df = build_query_classifier_frame(train_queries_raw_df)
test_queries_classifier_df = build_query_classifier_frame(test_queries_raw_df)

if docs_df["content"].eq("").all():
    raise ValueError("All document content is empty after preprocessing. Check the source columns or normalization.")

print(f"Average document length (chars): {docs_df['content'].str.len().mean():.1f}")
print(f"Average train query length      : {train_queries_df['content'].str.len().mean():.1f}")
print(f"Average test query length       : {test_queries_df['content'].str.len().mean():.1f}")


Average document length (chars): 900.9
Average train query length      : 48.4
Average test query length       : 48.9


## Retrieval Index Construction

The main engineering goal in this section is to make expensive work reusable. We cache document embeddings, TF-IDF artifacts, BM25 indexes, and the category classifier so repeated notebook runs do not recompute the same objects unnecessarily.

The dense retriever is the most expensive component, so caching and chunked scoring matter most there.


In [4]:
_MODEL_MEMORY_CACHE: dict[str, Any] = {}
_ARRAY_MEMORY_CACHE: dict[str, np.ndarray] = {}
_OBJECT_MEMORY_CACHE: dict[str, Any] = {}


def _safe_component(value: Any) -> str:
    """Make a string safe for use in cache file names."""
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value))


def _hash_payload(payload: dict[str, Any]) -> str:
    """Create a short deterministic hash for cache keys."""
    raw_payload = json.dumps(payload, sort_keys=True, ensure_ascii=True, default=str)
    return hashlib.sha1(raw_payload.encode("utf-8")).hexdigest()[:16]


def _normalization_signature() -> str:
    """Fingerprint the preprocessing configuration used by the retrievers."""
    payload = {
        "normalization": NORMALIZATION_CONFIG,
        "token_pattern": TOKEN_PATTERN,
    }
    return _hash_payload(payload)


def _dataframe_fingerprint(frame: pd.DataFrame, columns: Sequence[str]) -> str:
    """Fingerprint selected dataframe columns for cache invalidation."""
    hasher = hashlib.sha1()
    hasher.update(str(len(frame)).encode("utf-8"))
    for column in columns:
        hasher.update(column.encode("utf-8"))
        column_hash = pd.util.hash_pandas_object(frame[column].astype(str), index=False).values
        hasher.update(column_hash.tobytes())
    return hasher.hexdigest()[:16]


def _load_pickle(path: Path) -> Any:
    """Load a pickled artifact from disk."""
    with open(path, "rb") as handle:
        return pickle.load(handle)


def _save_pickle(path: Path, artifact: Any) -> None:
    """Persist an artifact to disk with the highest pickle protocol."""
    with open(path, "wb") as handle:
        pickle.dump(artifact, handle, protocol=pickle.HIGHEST_PROTOCOL)


def _load_sentence_model(model_name: str) -> Any:
    """Load a Sentence-Transformer model, preferring the local cache when available."""
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `sentence_transformers`. Install it with `%pip install sentence-transformers`."
        ) from exc

    if model_name in _MODEL_MEMORY_CACHE:
        return _MODEL_MEMORY_CACHE[model_name]

    safe_model_name = _safe_component(model_name)
    local_model_dir = MODEL_CACHE_DIR / safe_model_name
    if local_model_dir.exists():
        print(f"Loading model weights from cache: {local_model_dir}")
        model = SentenceTransformer(str(local_model_dir))
    else:
        print(f"Downloading model weights: {model_name}")
        model = SentenceTransformer(model_name)
        local_model_dir.mkdir(parents=True, exist_ok=True)
        model.save(str(local_model_dir))
        print(f"Saved model weights to cache: {local_model_dir}")

    _MODEL_MEMORY_CACHE[model_name] = model
    return model


def _load_or_encode_embeddings(
    frame: pd.DataFrame,
    kind: str,
    model: Any,
    model_name: str,
    batch_size: int,
) -> np.ndarray:
    """Load cached embeddings or encode them once and persist the result.

    Args:
        frame: Dataframe containing `id` and normalized `content`.
        kind: Human-readable cache prefix such as `docs` or `queries_train`.
        model: Loaded Sentence-Transformer model.
        model_name: Model identifier used in cache keys.
        batch_size: Sentence-Transformer encoding batch size.

    Returns:
        A float32 matrix of L2-normalized embeddings.
    """
    signature = _dataframe_fingerprint(frame, ["id", "content"])
    normalization_signature = _normalization_signature()
    cache_name = f"{kind}_{_safe_component(model_name)}_{normalization_signature}_{signature}.npy"
    cache_path = EMBEDDING_CACHE_DIR / cache_name
    memory_key = str(cache_path.resolve())

    if memory_key in _ARRAY_MEMORY_CACHE:
        return _ARRAY_MEMORY_CACHE[memory_key]

    if ENABLE_EMBEDDING_CACHE and cache_path.exists():
        print(f"Loading {kind} embeddings from cache: {cache_path.name}")
        embeddings = np.load(cache_path)
    else:
        print(f"Encoding {len(frame):,} {kind} rows...")
        embeddings = model.encode(
            frame["content"].tolist(),
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        embeddings = np.asarray(embeddings, dtype=np.float32)
        if ENABLE_EMBEDDING_CACHE:
            np.save(cache_path, embeddings)
            print(f"Saved {kind} embeddings to cache: {cache_path.name}")

    _ARRAY_MEMORY_CACHE[memory_key] = embeddings
    return embeddings


def _tfidf_param_candidates() -> list[dict[str, Any]]:
    """Return TF-IDF parameter settings including a safe fallback for very small corpora."""
    candidates = [dict(TFIDF_CONFIG)]
    min_df = TFIDF_CONFIG.get("min_df", 1)
    if isinstance(min_df, int) and min_df > 1:
        fallback = dict(TFIDF_CONFIG)
        fallback["min_df"] = 1
        candidates.append(fallback)
    return candidates


def build_or_load_tfidf_index(docs_frame: pd.DataFrame) -> dict[str, Any]:
    """Build or load the cached TF-IDF vectorizer and document matrix."""
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `scikit-learn`. Install it with `%pip install scikit-learn`."
        ) from exc

    docs_signature = _dataframe_fingerprint(docs_frame, ["id", "content"])
    normalization_signature = _normalization_signature()
    doc_ids = docs_frame["id"].to_numpy()

    for params in _tfidf_param_candidates():
        cache_key = _hash_payload(
            {
                "docs_signature": docs_signature,
                "normalization_signature": normalization_signature,
                "tfidf_params": params,
            }
        )
        cache_path = TFIDF_CACHE_DIR / f"tfidf_{cache_key}.pkl"
        memory_key = str(cache_path.resolve())

        if memory_key in _OBJECT_MEMORY_CACHE:
            return _OBJECT_MEMORY_CACHE[memory_key]
        if ENABLE_CLASSIC_CACHE and cache_path.exists():
            print(f"Loading TF-IDF artifacts from cache: {cache_path.name}")
            artifacts = _load_pickle(cache_path)
            _OBJECT_MEMORY_CACHE[memory_key] = artifacts
            return artifacts

    params = dict(TFIDF_CONFIG)
    vectorizer = TfidfVectorizer(**params)
    try:
        doc_vectors = vectorizer.fit_transform(docs_frame["content"])
    except ValueError as err:
        if "After pruning, no terms remain" not in str(err) or params.get("min_df", 1) == 1:
            raise
        params["min_df"] = 1
        vectorizer = TfidfVectorizer(**params)
        doc_vectors = vectorizer.fit_transform(docs_frame["content"])

    cache_key = _hash_payload(
        {
            "docs_signature": docs_signature,
            "normalization_signature": normalization_signature,
            "tfidf_params": params,
        }
    )
    cache_path = TFIDF_CACHE_DIR / f"tfidf_{cache_key}.pkl"
    memory_key = str(cache_path.resolve())
    artifacts = {
        "vectorizer": vectorizer,
        "doc_vectors": doc_vectors,
        "doc_ids": doc_ids,
        "params": params,
    }

    if ENABLE_CLASSIC_CACHE:
        _save_pickle(cache_path, artifacts)
        print(f"Saved TF-IDF artifacts to cache: {cache_path.name}")

    _OBJECT_MEMORY_CACHE[memory_key] = artifacts
    return artifacts


def build_or_load_bm25_index(docs_frame: pd.DataFrame) -> dict[str, Any]:
    """Build or load the cached BM25+ index."""
    try:
        from rank_bm25 import BM25Plus
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `rank_bm25`. Install it with `%pip install rank_bm25`."
        ) from exc

    docs_signature = _dataframe_fingerprint(docs_frame, ["id", "content"])
    normalization_signature = _normalization_signature()
    cache_key = _hash_payload(
        {
            "docs_signature": docs_signature,
            "normalization_signature": normalization_signature,
            "bm25_params": BM25_CONFIG,
        }
    )
    cache_path = BM25_CACHE_DIR / f"bm25_{cache_key}.pkl"
    memory_key = str(cache_path.resolve())

    if memory_key in _OBJECT_MEMORY_CACHE:
        return _OBJECT_MEMORY_CACHE[memory_key]
    if ENABLE_CLASSIC_CACHE and cache_path.exists():
        print(f"Loading BM25 index from cache: {cache_path.name}")
        artifacts = _load_pickle(cache_path)
        _OBJECT_MEMORY_CACHE[memory_key] = artifacts
        return artifacts

    tokenized_corpus = [tokenize(text) for text in docs_frame["content"]]
    bm25 = BM25Plus(tokenized_corpus, **BM25_CONFIG)
    artifacts = {
        "bm25": bm25,
        "doc_ids": docs_frame["id"].to_numpy(),
    }

    if ENABLE_CLASSIC_CACHE:
        _save_pickle(cache_path, artifacts)
        print(f"Saved BM25 index to cache: {cache_path.name}")

    _OBJECT_MEMORY_CACHE[memory_key] = artifacts
    return artifacts


def build_or_load_category_classifier(train_frame: pd.DataFrame) -> dict[str, Any]:
    """Build or load the cached TF-IDF + LinearSVC category classifier."""
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.svm import LinearSVC
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `scikit-learn`. Install it with `%pip install scikit-learn`."
        ) from exc

    require_columns(train_frame, ["id", "content", "category"], "Classifier training data")
    train_signature = _dataframe_fingerprint(train_frame, ["id", "content", "category"])
    normalization_signature = _normalization_signature()
    cache_key = _hash_payload(
        {
            "train_signature": train_signature,
            "normalization_signature": normalization_signature,
            "classifier_tfidf_params": CLASSIFIER_TFIDF_CONFIG,
        }
    )
    cache_path = CLASSIFIER_CACHE_DIR / f"category_classifier_{cache_key}.pkl"
    memory_key = str(cache_path.resolve())

    if memory_key in _OBJECT_MEMORY_CACHE:
        return _OBJECT_MEMORY_CACHE[memory_key]
    if ENABLE_CLASSIC_CACHE and cache_path.exists():
        print(f"Loading category classifier from cache: {cache_path.name}")
        artifacts = _load_pickle(cache_path)
        _OBJECT_MEMORY_CACHE[memory_key] = artifacts
        return artifacts

    vectorizer = TfidfVectorizer(**CLASSIFIER_TFIDF_CONFIG)
    train_vectors = vectorizer.fit_transform(train_frame["content"])
    classifier = LinearSVC()
    classifier.fit(train_vectors, train_frame["category"].astype(str))

    artifacts = {
        "vectorizer": vectorizer,
        "classifier": classifier,
    }

    if ENABLE_CLASSIC_CACHE:
        _save_pickle(cache_path, artifacts)
        print(f"Saved category classifier to cache: {cache_path.name}")

    _OBJECT_MEMORY_CACHE[memory_key] = artifacts
    return artifacts


## Retrieval Functions

The retrieval code is intentionally separated from index construction. That keeps the notebook easier to reason about and makes it straightforward to compare methods fairly.

The main performance changes in this refactor are:
- use **partial top-k selection** with `np.argpartition` instead of sorting every score vector fully
- score embedding queries in **chunks** to avoid creating unnecessarily large dense matrices
- reuse the **same max-k ranking** during offline sweeps and truncate it for smaller K values


In [5]:
def validate_pipeline_settings(document_count: int) -> None:
    """Validate global retrieval and submission settings."""
    if FINAL_MODEL not in {"tfidf", "bm25", "embedding"}:
        raise ValueError(f"Unknown FINAL_MODEL: {FINAL_MODEL}")
    if any(model_name not in {"tfidf", "bm25", "embedding"} for model_name in EVALUATION_MODELS):
        raise ValueError(f"Unknown model in EVALUATION_MODELS: {EVALUATION_MODELS}")
    if document_count <= 0:
        raise ValueError("The document collection is empty.")
    if SUBMIT_TOP_K <= 0:
        raise ValueError("SUBMIT_TOP_K must be positive.")


def top_k_indices(score_vector: np.ndarray, top_k: int) -> np.ndarray:
    """Return indices of the top-k scores in descending order.

    This uses `np.argpartition` to avoid a full sort when only the largest values are needed.
    """
    if top_k <= 0:
        raise ValueError("top_k must be positive.")

    capped_top_k = min(top_k, score_vector.shape[0])
    if capped_top_k == score_vector.shape[0]:
        return np.argsort(score_vector)[::-1]

    candidate_indices = np.argpartition(score_vector, -capped_top_k)[-capped_top_k:]
    sorted_candidates = candidate_indices[np.argsort(score_vector[candidate_indices])[::-1]]
    return sorted_candidates


def truncate_results(results: list[RetrievalResult], top_k: int) -> list[RetrievalResult]:
    """Truncate a ranked result list to a smaller K without recomputing scores."""
    if top_k <= 0:
        raise ValueError("top_k must be positive.")
    return [
        {
            "query_id": result["query_id"],
            "relevant_docs": result["relevant_docs"][:top_k],
        }
        for result in results
    ]


def progress_interval(total_items: int, target_updates: int = 5) -> int:
    """Choose a lightweight logging interval for progress messages."""
    return max(1, total_items // max(1, target_updates))


def prepare_retriever(model_name: ModelName, docs_frame: pd.DataFrame) -> dict[str, Any]:
    """Prepare and cache the artifacts required by one retriever."""
    if model_name == "tfidf":
        return build_or_load_tfidf_index(docs_frame)
    if model_name == "bm25":
        return build_or_load_bm25_index(docs_frame)
    if model_name == "embedding":
        model = _load_sentence_model(EMBEDDING_MODEL_NAME)
        doc_embeddings = _load_or_encode_embeddings(
            docs_frame,
            kind="docs",
            model=model,
            model_name=EMBEDDING_MODEL_NAME,
            batch_size=EMBEDDING_BATCH_SIZE,
        )
        return {
            "model": model,
            "doc_embeddings": doc_embeddings,
            "doc_ids": docs_frame["id"].to_numpy(),
        }
    raise ValueError(f"Unknown model: {model_name}")


def run_tfidf_search(
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    top_k: int,
    prepared_artifacts: dict[str, Any] | None = None,
) -> list[RetrievalResult]:
    """Run sparse lexical retrieval with TF-IDF cosine similarity."""
    artifacts = prepared_artifacts or build_or_load_tfidf_index(docs_frame)
    vectorizer = artifacts["vectorizer"]
    doc_vectors = artifacts["doc_vectors"]
    doc_ids = artifacts["doc_ids"]
    query_vectors = vectorizer.transform(queries_frame["content"])
    query_ids = queries_frame["id"].astype(str).tolist()
    capped_top_k = min(top_k, len(doc_ids))
    log_every = progress_interval(len(query_ids))

    print(
        f"  [TF-IDF] vectorized {len(query_ids):,} queries against {len(doc_ids):,} docs "
        f"with capped_top_k={capped_top_k:,}"
    )

    results: list[RetrievalResult] = []
    for row_index, query_id in enumerate(query_ids):
        score_row = query_vectors[row_index] @ doc_vectors.T
        score_vector = np.asarray(score_row.toarray()).ravel()
        top_indices = top_k_indices(score_vector, capped_top_k)
        results.append(
            {
                "query_id": query_id,
                "relevant_docs": doc_ids[top_indices].tolist(),
            }
        )
        if (row_index + 1) % log_every == 0 or row_index == len(query_ids) - 1:
            print(f"  [TF-IDF] processed {row_index + 1:,}/{len(query_ids):,} queries")
    return results


def run_bm25_search(
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    top_k: int,
    prepared_artifacts: dict[str, Any] | None = None,
) -> list[RetrievalResult]:
    """Run lexical retrieval with BM25+ over tokenized text."""
    artifacts = prepared_artifacts or build_or_load_bm25_index(docs_frame)
    bm25 = artifacts["bm25"]
    doc_ids = artifacts["doc_ids"]
    capped_top_k = min(top_k, len(doc_ids))
    query_pairs = list(queries_frame[["id", "content"]].itertuples(index=False, name=None))
    log_every = progress_interval(len(query_pairs))

    print(
        f"  [BM25+] scoring {len(query_pairs):,} queries against {len(doc_ids):,} docs "
        f"with capped_top_k={capped_top_k:,}"
    )

    results: list[RetrievalResult] = []
    for row_index, (query_id, query_text) in enumerate(query_pairs):
        score_vector = np.asarray(bm25.get_scores(tokenize(query_text)), dtype=np.float32)
        top_indices = top_k_indices(score_vector, capped_top_k)
        results.append(
            {
                "query_id": str(query_id),
                "relevant_docs": doc_ids[top_indices].tolist(),
            }
        )
        if (row_index + 1) % log_every == 0 or row_index == len(query_pairs) - 1:
            print(f"  [BM25+] processed {row_index + 1:,}/{len(query_pairs):,} queries")
    return results


def run_embedding_search(
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    top_k: int,
    prepared_artifacts: dict[str, Any] | None = None,
    embedding_kind: str = "queries",
) -> list[RetrievalResult]:
    """Run dense semantic retrieval with Sentence-Transformer embeddings."""
    artifacts = prepared_artifacts or prepare_retriever("embedding", docs_frame)
    model = artifacts["model"]
    doc_embeddings = artifacts["doc_embeddings"]
    doc_ids = artifacts["doc_ids"]
    query_embeddings = _load_or_encode_embeddings(
        queries_frame,
        kind=embedding_kind,
        model=model,
        model_name=EMBEDDING_MODEL_NAME,
        batch_size=EMBEDDING_BATCH_SIZE,
    )

    query_ids = queries_frame["id"].astype(str).tolist()
    capped_top_k = min(top_k, len(doc_ids))
    results: list[RetrievalResult] = []
    total_chunks = (len(query_embeddings) + EMBEDDING_QUERY_CHUNK_SIZE - 1) // EMBEDDING_QUERY_CHUNK_SIZE

    print(
        f"  [Embedding] scoring {len(query_ids):,} queries against {len(doc_ids):,} docs "
        f"with capped_top_k={capped_top_k:,}, chunk_size={EMBEDDING_QUERY_CHUNK_SIZE:,}, "
        f"embedding_cache_key='{embedding_kind}'"
    )

    for chunk_index, start_index in enumerate(range(0, len(query_embeddings), EMBEDDING_QUERY_CHUNK_SIZE), start=1):
        stop_index = start_index + EMBEDDING_QUERY_CHUNK_SIZE
        print(
            f"  [Embedding] chunk {chunk_index:,}/{total_chunks:,}: "
            f"queries {start_index + 1:,}-{min(stop_index, len(query_embeddings)):,}"
        )
        score_block = query_embeddings[start_index:stop_index] @ doc_embeddings.T
        for row_offset, score_vector in enumerate(score_block):
            top_indices = top_k_indices(score_vector, capped_top_k)
            query_id = query_ids[start_index + row_offset]
            results.append(
                {
                    "query_id": query_id,
                    "relevant_docs": doc_ids[top_indices].tolist(),
                }
            )
    return results


MODELS: dict[ModelName, Any] = {
    "tfidf": run_tfidf_search,
    "bm25": run_bm25_search,
    "embedding": run_embedding_search,
}


def run_retrieval(
    model_name: ModelName,
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    top_k: int,
    prepared_artifacts: dict[str, Any] | None = None,
    embedding_kind: str = "queries",
) -> list[RetrievalResult]:
    """Dispatch retrieval to the selected method and log the runtime."""
    if model_name not in MODELS:
        raise ValueError(f"Unknown model '{model_name}'. Choose from {list(MODELS)}")

    print("=" * 88)
    print(f"Starting retrieval: model={model_name}")
    print(
        f"  parameters: top_k={top_k:,}, docs={len(docs_frame):,}, queries={len(queries_frame):,}, "
        f"prepared_artifacts={'yes' if prepared_artifacts is not None else 'no'}, "
        f"embedding_kind='{embedding_kind}'"
    )
    start_time = time.time()
    if model_name == "embedding":
        results = MODELS[model_name](
            docs_frame,
            queries_frame,
            top_k=top_k,
            prepared_artifacts=prepared_artifacts,
            embedding_kind=embedding_kind,
        )
    else:
        results = MODELS[model_name](
            docs_frame,
            queries_frame,
            top_k=top_k,
            prepared_artifacts=prepared_artifacts,
        )
    elapsed_seconds = time.time() - start_time
    print(
        f"Completed retrieval: model={model_name}, results={len(results):,} queries, "
        f"elapsed={elapsed_seconds:.1f}s"
    )
    print("=" * 88)
    return results
validate_pipeline_settings(document_count=len(docs_df))
RETRIEVAL_TOP_K = SUBMIT_TOP_K
print(f"Submission top_k : {SUBMIT_TOP_K:,}")
print(f"Retrieval top_k  : {RETRIEVAL_TOP_K:,}")


Submission top_k : 7,500
Retrieval top_k  : 7,500


## Evaluation Logic

Offline evaluation mirrors the competition-style retrieval objective. The notebook reports:
- **Recall@K**: how much of the relevant set is recovered
- **Precision@K**: how much of the returned list is relevant
- **MRR@K**: how early the first relevant document appears
- **Accuracy**: category prediction accuracy from the lightweight query classifier

The combined offline score is the simple average of those four values.


In [6]:
def load_ground_truth(path: Path) -> dict[str, GroundTruthEntry]:
    """Load the training relevance annotations from `qgts_train.json`."""
    if not path.exists():
        raise FileNotFoundError(f"Ground-truth file not found: {path}")

    with open(path, "r", encoding="utf-8") as handle:
        raw_ground_truth = json.load(handle)

    ground_truth: dict[str, GroundTruthEntry] = {}
    for query_id, info in raw_ground_truth.items():
        relevant_items = info.get("relevant_doc_ids", [])
        ground_truth[str(query_id)] = {
            "relevant_doc_ids": {str(item["doc_id"]) for item in relevant_items},
            "total_relevant_docs": int(info.get("total_relevant_docs", len(relevant_items))),
            "category": info.get("category"),
        }
    return ground_truth


def recall_at_k(results: list[RetrievalResult], ground_truth: dict[str, GroundTruthEntry], k: int) -> float:
    """Compute mean Recall@K across all queries present in the ground truth."""
    recalls: list[float] = []
    for item in results:
        query_id = str(item["query_id"])
        if query_id not in ground_truth:
            continue

        relevant_doc_ids = ground_truth[query_id]["relevant_doc_ids"]
        total_relevant_docs = ground_truth[query_id]["total_relevant_docs"]
        predicted_doc_ids = item["relevant_docs"][:k]
        hits = sum(doc_id in relevant_doc_ids for doc_id in predicted_doc_ids)
        recall_value = hits / total_relevant_docs if total_relevant_docs > 0 else 0.0
        recalls.append(recall_value)

    return float(np.mean(recalls)) if recalls else 0.0


def precision_at_k(results: list[RetrievalResult], ground_truth: dict[str, GroundTruthEntry], k: int) -> float:
    """Compute mean Precision@K across all queries present in the ground truth."""
    precisions: list[float] = []
    for item in results:
        query_id = str(item["query_id"])
        if query_id not in ground_truth:
            continue

        relevant_doc_ids = ground_truth[query_id]["relevant_doc_ids"]
        predicted_doc_ids = item["relevant_docs"][:k]
        if not predicted_doc_ids:
            precisions.append(0.0)
            continue

        hits = sum(doc_id in relevant_doc_ids for doc_id in predicted_doc_ids)
        precisions.append(hits / len(predicted_doc_ids))

    return float(np.mean(precisions)) if precisions else 0.0


def mrr_at_k(results: list[RetrievalResult], ground_truth: dict[str, GroundTruthEntry], k: int) -> float:
    """Compute mean reciprocal rank at K."""
    reciprocal_ranks: list[float] = []
    for item in results:
        query_id = str(item["query_id"])
        if query_id not in ground_truth:
            continue

        relevant_doc_ids = ground_truth[query_id]["relevant_doc_ids"]
        reciprocal_rank = 0.0
        for rank, doc_id in enumerate(item["relevant_docs"][:k], start=1):
            if doc_id in relevant_doc_ids:
                reciprocal_rank = 1.0 / rank
                break
        reciprocal_ranks.append(reciprocal_rank)

    return float(np.mean(reciprocal_ranks)) if reciprocal_ranks else 0.0


def compute_category_accuracy(
    ground_truth: dict[str, GroundTruthEntry],
    predicted_categories: dict[str, str] | None,
    default_if_missing: float = 0.0,
) -> float:
    """Compute query category accuracy when category predictions are available."""
    if predicted_categories is None:
        return float(default_if_missing)

    try:
        from sklearn.metrics import accuracy_score
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `scikit-learn`. Install it with `%pip install scikit-learn`."
        ) from exc

    y_true: list[str] = []
    y_pred: list[str] = []
    for query_id, info in ground_truth.items():
        true_category = info.get("category")
        predicted_category = predicted_categories.get(str(query_id))
        if true_category is None or predicted_category is None:
            continue
        y_true.append(str(true_category))
        y_pred.append(str(predicted_category))

    if not y_true:
        return float(default_if_missing)
    return float(accuracy_score(y_true, y_pred))


def leaderboard_score(
    results: list[RetrievalResult],
    ground_truth: dict[str, GroundTruthEntry],
    k: int,
    predicted_categories: dict[str, str] | None = None,
    accuracy_value: float | None = None,
) -> dict[str, float]:
    """Compute the combined offline leaderboard-style score."""
    recall_value = recall_at_k(results, ground_truth, k=k)
    precision_value = precision_at_k(results, ground_truth, k=k)
    mrr_value = mrr_at_k(results, ground_truth, k=k)
    category_accuracy = (
        float(accuracy_value)
        if accuracy_value is not None
        else compute_category_accuracy(ground_truth, predicted_categories)
    )
    combined_score = 0.25 * (recall_value + precision_value + mrr_value + category_accuracy)
    return {
        "Recall": recall_value,
        "Precision": precision_value,
        "MRR": mrr_value,
        "Accuracy": category_accuracy,
        "LeaderboardScore": combined_score,
    }


def predict_category_map(query_frame: pd.DataFrame, classifier_artifacts: dict[str, Any]) -> dict[str, str]:
    """Predict one category label per query using the cached classifier."""
    query_vectors = classifier_artifacts["vectorizer"].transform(query_frame["content"])
    predictions = classifier_artifacts["classifier"].predict(query_vectors)
    return {
        str(query_id): str(prediction)
        for query_id, prediction in zip(query_frame["id"].astype(str), predictions)
    }


def build_doc_category_map(docs_frame: pd.DataFrame) -> dict[str, Any]:
    """Build a lookup from document id to document category."""
    require_columns(docs_frame, ["id", "category"], "Documents frame")
    return (
        docs_frame[["id", "category"]]
        .assign(id=lambda frame: frame["id"].astype(str))
        .set_index("id")["category"]
        .to_dict()
    )


def filter_results_by_dominant_category(
    results: list[RetrievalResult],
    doc_category_map: dict[str, Any],
    dominant_top_n: int = DOMINANT_CATEGORY_TOP_N,
) -> list[RetrievalResult]:
    """Keep only docs from the dominant category in each query's top-N window."""
    if dominant_top_n <= 0:
        raise ValueError("dominant_top_n must be positive.")

    filtered_results: list[RetrievalResult] = []
    for item in results:
        query_id = str(item["query_id"])
        doc_ids = [str(doc_id) for doc_id in item["relevant_docs"]]
        top_window = doc_ids[:dominant_top_n]
        if not top_window:
            filtered_results.append({"query_id": query_id, "relevant_docs": []})
            continue

        category_counts: dict[str, int] = {}
        first_position: dict[str, int] = {}
        for position, doc_id in enumerate(top_window):
            raw_category = doc_category_map.get(doc_id)
            category = "unknown" if raw_category is None or pd.isna(raw_category) else str(raw_category)
            category_counts[category] = category_counts.get(category, 0) + 1
            if category not in first_position:
                first_position[category] = position

        dominant_category = min(
            category_counts,
            key=lambda category: (-category_counts[category], first_position[category], category),
        )
        filtered_docs = []
        for doc_id in doc_ids:
            raw_category = doc_category_map.get(doc_id)
            category = "unknown" if raw_category is None or pd.isna(raw_category) else str(raw_category)
            if category == dominant_category:
                filtered_docs.append(doc_id)
        filtered_results.append({"query_id": query_id, "relevant_docs": filtered_docs})
    return filtered_results


def write_kaggle_submission(
    results: list[RetrievalResult],
    sample_csv_path: Path,
    output_csv_path: Path,
    category_predictions: dict[str, str] | None = None,
) -> None:
    """Write predictions in the exact Kaggle submission format."""
    prediction_map = {
        str(item["query_id"]): [str(doc_id) for doc_id in item["relevant_docs"]]
        for item in results
    }

    with open(sample_csv_path, "r", newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        rows = list(reader)

    if fieldnames is None or len(fieldnames) < 2:
        raise ValueError("Invalid sample submission format.")

    query_id_column = fieldnames[0]
    prediction_column = fieldnames[1]
    category_column = fieldnames[2] if len(fieldnames) >= 3 else None

    with open(output_csv_path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            query_id = str(row[query_id_column])
            if query_id not in prediction_map:
                raise ValueError(f"Missing retrieval prediction for query_id={query_id}")

            output_row = {
                query_id_column: query_id,
                prediction_column: json.dumps(prediction_map[query_id]),
            }
            if category_column is not None:
                if category_predictions is None:
                    output_row[category_column] = row.get(category_column, "?") or "?"
                else:
                    if query_id not in category_predictions:
                        raise ValueError(f"Missing category prediction for query_id={query_id}")
                    output_row[category_column] = str(category_predictions[query_id])
            writer.writerow(output_row)


ground_truth = load_ground_truth(ground_truth_path)
print(f"Ground-truth queries: {len(ground_truth):,}")


Ground-truth queries: 327


## Interpreting Recall, Precision, MRR, and Accuracy

The four reported metrics answer different questions:

- **Recall@K**: “Did we recover most of the relevant documents at all?”
- **Precision@K**: “How much noise is in the returned list?”
- **MRR@K**: “How early does the first useful document appear?”
- **Accuracy**: “Did the classifier predict the correct category label?”

Interpretation tips:
- If recall is low, the first-stage retriever is failing to surface relevant candidates.
- If recall is high but MRR is low, the right documents are present but ranked too deep.
- If precision falls sharply as K grows, the extra tail is mostly noise.
- If accuracy is stable across runs while retrieval metrics move, the changes are coming from the retriever rather than the classifier.


## Practical Tradeoffs in the Pipeline

A few practical engineering lessons matter more than clever modeling here:

- **Cache expensive artifacts**. Dense embeddings and fitted indexes are too expensive to rebuild every run.
- **Keep preprocessing shared**. Small text-normalization differences across methods make comparisons noisy and harder to trust.
- **Optimize the bottlenecks first**. In this notebook the biggest wins come from avoiding repeated scoring work and full score sorting.
- **Improve the first-stage retriever first**. If the recall ceiling is weak, later refinements will not rescue the pipeline.
- **Use lexical methods intentionally**. They remain useful for exact strings, error codes, product names, and very tight latency budgets.


## Experiments

The experiment loop below keeps the current notebook behavior but makes the logic more explicit:
- train the category classifier once
- prepare retrieval artifacts once per model
- retrieve a **single max-k ranking** per model on the train queries
- reuse that ranking for smaller K values by truncation

Reusing the max-k ranking is a safe optimization because the top-`k_small` prefix of a correctly sorted top-`k_large` ranking is identical to rerunning retrieval directly at `k_small`.


In [7]:
category_classifier = build_or_load_category_classifier(docs_classifier_df)
train_query_category_map = predict_category_map(train_queries_classifier_df, category_classifier)
test_query_category_map = predict_category_map(test_queries_classifier_df, category_classifier)
classifier_accuracy = compute_category_accuracy(ground_truth, train_query_category_map)

print(f"Classifier accuracy on train queries: {classifier_accuracy:.4f}")
print(f"Train category predictions         : {len(train_query_category_map):,}")
print(f"Test category predictions          : {len(test_query_category_map):,}")

TOP_CATEGORY_STATS_K = 20
QUERY_CATEGORY_STATS_LIMIT = 5
doc_category_map = build_doc_category_map(docs_df)

def format_category_percentages(doc_ids: list[str], doc_categories: dict[str, Any]) -> str:
    """Format category percentages for one query's retrieved document ids."""
    category_counts: dict[str, int] = {}
    total_docs = 0

    for doc_id in doc_ids:
        raw_category = doc_categories.get(str(doc_id))
        if raw_category is None or pd.isna(raw_category):
            category = "unknown"
        else:
            category = str(raw_category)
        category_counts[category] = category_counts.get(category, 0) + 1
        total_docs += 1

    if total_docs == 0:
        return ""

    sorted_counts = sorted(category_counts.items(), key=lambda item: (-item[1], item[0]))
    return " | ".join(
        f"{category}: {100.0 * count / total_docs:.2f}%"
        for category, count in sorted_counts
    )


def top_category_percentages_by_query(
    results: list[RetrievalResult],
    doc_categories: dict[str, Any],
    top_n: int = TOP_CATEGORY_STATS_K,
) -> dict[str, str]:
    """Return one category-percentage summary per query."""
    return {
        str(item["query_id"]): format_category_percentages(item["relevant_docs"][:top_n], doc_categories)
        for item in results
    }

prepared_retrievers: dict[ModelName, dict[str, Any]] = {}
for model_name in set(EVALUATION_MODELS) | {FINAL_MODEL}:
    print(f"Preparing artifacts for {model_name}...")
    prepared_retrievers[model_name] = prepare_retriever(model_name, docs_df)

max_eval_top_k = max(EVALUATION_TOP_KS)
evaluation_rows: list[dict[str, Any]] = []
evaluation_query_rows: list[dict[str, Any]] = []

for model_name in EVALUATION_MODELS:
    max_k_results = run_retrieval(
        model_name=model_name,
        docs_frame=docs_df,
        queries_frame=train_queries_df,
        top_k=max_eval_top_k,
        prepared_artifacts=prepared_retrievers[model_name],
        embedding_kind="queries_train",
    )
    if ENABLE_CATEGORY_FILTER:
        max_k_results = filter_results_by_dominant_category(
            max_k_results,
            doc_category_map=doc_category_map,
            dominant_top_n=DOMINANT_CATEGORY_TOP_N,
        )
        kept_counts = [len(item["relevant_docs"]) for item in max_k_results]
        zero_doc_queries = sum(count == 0 for count in kept_counts)
        print(
            "  [DominantTopNFilter] "
            f"top_n={DOMINANT_CATEGORY_TOP_N}, "
            f"kept_docs_per_query(min/mean/max)="
            f"{min(kept_counts):,}/{float(np.mean(kept_counts)):.1f}/{max(kept_counts):,}, "
            f"zero_doc_queries={zero_doc_queries:,}/{len(kept_counts):,}"
        )

    for top_k in EVALUATION_TOP_KS:
        truncated_results = truncate_results(max_k_results, top_k)
        metrics = leaderboard_score(
            truncated_results,
            ground_truth,
            k=top_k,
            accuracy_value=classifier_accuracy,
        )
        limited_results = truncated_results[:QUERY_CATEGORY_STATS_LIMIT]
        top20_category_pct_by_query = top_category_percentages_by_query(
            limited_results,
            doc_categories=doc_category_map,
            top_n=TOP_CATEGORY_STATS_K,
        )
        evaluation_rows.append(
            {
                "Model": model_name,
                "TopK": top_k,
                **metrics,
            }
        )
        for query_id, top20_category_pct in top20_category_pct_by_query.items():
            evaluation_query_rows.append(
                {
                    "Model": model_name,
                    "TopK": top_k,
                    "QueryID": query_id,
                    "Top20CategoryPct": top20_category_pct,
                }
            )
        print(
            f"{model_name:10s} top_k={top_k:>6,}  "
            f"Recall={metrics['Recall']:.5f}  "
            f"Precision={metrics['Precision']:.5f}  "
            f"MRR={metrics['MRR']:.5f}  "
            f"Accuracy={metrics['Accuracy']:.5f}  "
            f"Score={metrics['LeaderboardScore']:.5f}  "
            f"Top{TOP_CATEGORY_STATS_K}CatsByQuery={len(top20_category_pct_by_query):,} "
            f"(first {QUERY_CATEGORY_STATS_LIMIT} queries)"
        )

eval_summary_df = (
    pd.DataFrame(evaluation_rows)
    .sort_values(["LeaderboardScore", "TopK"], ascending=[False, False])
    .reset_index(drop=True)
)

if eval_summary_df.empty:
    raise ValueError("No evaluation rows were produced.")

if not evaluation_query_rows:
    raise ValueError("No per-query category rows were produced.")

eval_df = (
    pd.DataFrame(evaluation_query_rows)
    .merge(eval_summary_df, on=["Model", "TopK"], how="left")
    .sort_values(["LeaderboardScore", "TopK", "QueryID"], ascending=[False, False, True])
    .reset_index(drop=True)
)

best_experiment = eval_summary_df.iloc[0]
print("\nBest offline configuration:")
print(best_experiment.to_string())
eval_df


Saved category classifier to cache: category_classifier_80fbb18d392b5e02.pkl
Classifier accuracy on train queries: 0.9266
Train category predictions         : 327
Test category predictions          : 141
Preparing artifacts for embedding...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model weights to cache: /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project/cache/sentence_transformers/all-MiniLM-L6-v2
Encoding 216,041 docs rows...


Batches:   0%|          | 0/844 [00:00<?, ?it/s]

Saved docs embeddings to cache: docs_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_0dc7d23bf7e28863.npy
Starting retrieval: model=embedding
  parameters: top_k=7,500, docs=216,041, queries=327, prepared_artifacts=yes, embedding_kind='queries_train'
Encoding 327 queries_train rows...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Saved queries_train embeddings to cache: queries_train_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_fcd04dbbbee9cd4f.npy
  [Embedding] scoring 327 queries against 216,041 docs with capped_top_k=7,500, chunk_size=32, embedding_cache_key='queries_train'
  [Embedding] chunk 1/11: queries 1-32
  [Embedding] chunk 2/11: queries 33-64
  [Embedding] chunk 3/11: queries 65-96
  [Embedding] chunk 4/11: queries 97-128
  [Embedding] chunk 5/11: queries 129-160
  [Embedding] chunk 6/11: queries 161-192
  [Embedding] chunk 7/11: queries 193-224
  [Embedding] chunk 8/11: queries 225-256
  [Embedding] chunk 9/11: queries 257-288
  [Embedding] chunk 10/11: queries 289-320
  [Embedding] chunk 11/11: queries 321-327
Completed retrieval: model=embedding, results=327 queries, elapsed=5.0s
  [DominantTopNFilter] top_n=20, kept_docs_per_query(min/mean/max)=988/4841.2/7,294, zero_doc_queries=0/327
embedding  top_k= 7,500  Recall=0.91572  Precision=0.00178  MRR=0.46989  Accuracy=0.92661  Score=0.57850  Top20CatsByQuery=

,Model,TopK,QueryID,Top20CategoryPct,Recall,Precision,MRR,Accuracy,LeaderboardScore
0,embedding,7500,3e66798a-b7fd-41b5-8bc0-33b3d7ce2aca_177487,tex: 100.00%,0.91572,0.00178,0.46989,0.92661,0.5785
1,embedding,7500,4008ed78-e66e-4d89-9c3b-c79bd1cf6fc9_366,unix: 100.00%,0.91572,0.00178,0.46989,0.92661,0.5785
2,embedding,7500,961c4349-8cf1-4ef1-89cc-24d20bb9d000_67878,android: 100.00%,0.91572,0.00178,0.46989,0.92661,0.5785
3,embedding,7500,d5a95b09-e8ea-44dd-993d-347ed418e1f1_15138,android: 100.00%,0.91572,0.00178,0.46989,0.92661,0.5785
4,embedding,7500,f5f944d2-277a-481d-ab09-612890402ded_137489,gaming: 100.00%,0.91572,0.00178,0.46989,0.92661,0.5785


## Generate Test Submission

The final submission uses the configured `FINAL_MODEL` and writes the first-stage retrieval ranking directly with `SUBMIT_TOP_K` documents per query.


In [8]:
final_model_artifacts = prepared_retrievers.get(FINAL_MODEL)
if final_model_artifacts is None:
    final_model_artifacts = prepare_retriever(FINAL_MODEL, docs_df)
    prepared_retrievers[FINAL_MODEL] = final_model_artifacts

test_results = run_retrieval(
    model_name=FINAL_MODEL,
    docs_frame=docs_df,
    queries_frame=test_queries_df,
    top_k=RETRIEVAL_TOP_K,
    prepared_artifacts=final_model_artifacts,
    embedding_kind="queries_test",
)
if ENABLE_CATEGORY_FILTER:
    test_results = filter_results_by_dominant_category(
        test_results,
        doc_category_map=build_doc_category_map(docs_df),
        dominant_top_n=DOMINANT_CATEGORY_TOP_N,
    )
    kept_counts = [len(item["relevant_docs"]) for item in test_results]
    zero_doc_queries = sum(count == 0 for count in kept_counts)
    print(
        "[DominantTopNFilter:test] "
        f"top_n={DOMINANT_CATEGORY_TOP_N}, "
        f"kept_docs_per_query(min/mean/max)="
        f"{min(kept_counts):,}/{float(np.mean(kept_counts)):.1f}/{max(kept_counts):,}, "
        f"zero_doc_queries={zero_doc_queries:,}/{len(kept_counts):,}"
    )
write_kaggle_submission(
    test_results,
    sample_csv_path=sample_submission_path,
    output_csv_path=PATHS.output_path,
    category_predictions=test_query_category_map,
)

submission_preview = pd.read_csv(PATHS.output_path)
print(f"Saved submission to: {PATHS.output_path.resolve()}")
print(f"Rows: {len(submission_preview):,}")
submission_preview.head()


Starting retrieval: model=embedding
  parameters: top_k=7,500, docs=216,041, queries=141, prepared_artifacts=yes, embedding_kind='queries_test'
Encoding 141 queries_test rows...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Saved queries_test embeddings to cache: queries_test_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_61b9042ee990036f.npy
  [Embedding] scoring 141 queries against 216,041 docs with capped_top_k=7,500, chunk_size=32, embedding_cache_key='queries_test'
  [Embedding] chunk 1/5: queries 1-32
  [Embedding] chunk 2/5: queries 33-64
  [Embedding] chunk 3/5: queries 65-96
  [Embedding] chunk 4/5: queries 97-128
  [Embedding] chunk 5/5: queries 129-141
Completed retrieval: model=embedding, results=141 queries, elapsed=2.1s
[DominantTopNFilter:test] top_n=20, kept_docs_per_query(min/mean/max)=1,462/5048.7/7,234, zero_doc_queries=0/141
Saved submission to: /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project/solutions_SeaFour.csv
Rows: 141


,query_id,relevant_doc_ids,category
0,4ffe16bc-5235-418d-9bf3-22d1f2c5796e_145437,"[""c583f3cd-b1ca-4b74-ac3e-1c6771eb6a8e_131332"", ""fd449cc9-3a39-4669-9c88-257f806ac488_108240"", ""659588e9-0080-47ec-b...",programmers
1,1bb2bb20-7f45-4dcf-a94a-420c454f87b8_56473,"[""edd58474-cef1-4d75-b184-99ee27def6a2_116280"", ""3fd76a85-c69c-47de-b4b6-5dedf515b0a2_30246"", ""82144446-dc83-4a54-81...",unix
2,6a9a342c-1275-4bb3-a818-8bcce53fac4f_34507,"[""43fa6e5a-6ad6-4701-ba44-68b513409ffc_17053"", ""13e0472b-720c-45c6-9cb5-77009740ef72_12626"", ""c989df0c-14b8-40f2-a2c...",android
3,cb216e47-add6-41fd-974a-39251e4df3aa_6777,"[""ef46acba-fed8-4bc1-ba2e-df09048d8743_27744"", ""4eb42420-65d5-4f57-9e4c-f79fe64ab921_46670"", ""65a10367-e197-471c-8ed...",unix
4,14f1d3f5-8271-400e-9ef2-8319de25c9a1_200748,"[""59108dd2-5f5a-4add-b947-3303c8aaa891_123331"", ""4ff296e2-e448-4f41-8e3d-4b56ac994a17_14385"", ""5a29f2de-54fa-4bdb-bc...",tex


## Conclusions

The notebook now reads as a reusable retrieval-engine report rather than a sequence of ad hoc cells.

The key practical takeaway remains the same: **embedding retrieval is the strongest default first-stage method for this dataset** because it is far more robust to semantic mismatch than TF-IDF or BM25+, while the lexical methods still remain useful baselines for exact-match-heavy scenarios and tight latency budgets.
